# Random Forest Project – Bank Marketing Dataset

In this project, we will be working with the **Bank Marketing Dataset** from a Portuguese banking institution.
The dataset is related to direct phone call marketing campaigns aimed at getting clients to subscribe to a **term deposit**.

Our goal is to **predict whether a client will subscribe to a term deposit** (target variable `y`: yes/no).

### Column Descriptions:
| Column | Description |
|---|---|
| age | Age of the client |
| job | Type of job |
| marital | Marital status |
| education | Level of education |
| default | Has credit in default? |
| balance | Average yearly balance (euros) |
| housing | Has housing loan? |
| loan | Has personal loan? |
| contact | Contact communication type |
| day | Last contact day of the month |
| month | Last contact month |
| duration | Last contact duration (seconds) |
| campaign | Number of contacts during this campaign |
| pdays | Days since last contact from previous campaign |
| previous | Number of contacts before this campaign |
| poutcome | Outcome of the previous marketing campaign |
| **y** | **Has the client subscribed to a term deposit? (Target)** |

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

## Get the Data

**Use pandas to read `bank-full.csv` as a dataframe called `bank`.**

In [ ]:
bank = pd.read_csv('bank-full.csv', sep=';')

**Check out the `info()`, `head()`, and `describe()` methods on the dataframe.**

In [ ]:
bank.info()

In [ ]:
bank.head()

In [ ]:
bank.describe()

## Exploratory Data Analysis

Let's do some data visualization using seaborn and pandas built-in plotting capabilities.

**Create a histogram of the `age` distribution, colored by the subscription outcome `y`.**

In [ ]:
plt.figure(figsize=(10,5))
bank[bank['y']=='no']['age'].hist(bins=30, alpha=0.6, color='blue', label='Not Subscribed')
bank[bank['y']=='yes']['age'].hist(bins=30, alpha=0.6, color='green', label='Subscribed')
plt.xlabel('Age')
plt.ylabel('Count')
plt.title('Age Distribution by Subscription Outcome')
plt.legend()
plt.show()

**Create a countplot showing the counts of subscriptions by job type, with hue defined by `y`.**

In [ ]:
plt.figure(figsize=(14,5))
sns.countplot(x='job', data=bank, hue='y', palette='Set1')
plt.xticks(rotation=45)
plt.title('Subscription Count by Job Type')
plt.tight_layout()
plt.show()

**Create a countplot showing subscription counts by marital status.**

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='marital', data=bank, hue='y', palette='Set2')
plt.title('Subscription Count by Marital Status')
plt.show()

**Create a boxplot to see the relationship between balance and subscription outcome.**

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='y', y='balance', data=bank, palette='coolwarm')
plt.title('Account Balance by Subscription Outcome')
plt.show()

**Let's look at the overall class distribution of the target variable `y`.**

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='y', data=bank, palette='Set1')
plt.title('Target Variable Distribution (Subscribed vs Not Subscribed)')
plt.show()

print(bank['y'].value_counts())

## Setting Up the Data

Let's prepare our data for the Random Forest Classification Model.

**Check `bank.info()` again.**

In [ ]:
bank.info()

## Categorical Features

Notice that several columns are categorical (job, marital, education, default, housing, loan, contact, month, poutcome).
We need to convert them to dummy variables so sklearn can understand them.

**Create a list of all categorical feature column names called `cat_feats`.**

In [ ]:
cat_feats = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

**Use `pd.get_dummies()` to create dummy variables. Set this as `final_data`.**

Also convert the target variable `y` from 'yes'/'no' to 1/0.

In [ ]:
final_data = pd.get_dummies(bank, columns=cat_feats, drop_first=True)

# Convert target variable to binary
final_data['y'] = (final_data['y'] == 'yes').astype(int)

print('Shape after encoding:', final_data.shape)
final_data.head()

## Train Test Split

**Split the data into training and testing sets.**

In [ ]:
from sklearn.model_selection import train_test_split

X = final_data.drop('y', axis=1)
y = final_data['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

print('Training set size:', X_train.shape)
print('Testing set size:', X_test.shape)

## Training a Decision Tree Model

Let's start by training a single decision tree first!

**Import `DecisionTreeClassifier`.**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

**Create an instance of `DecisionTreeClassifier()` called `dtree` and fit it to the training data.**

In [ ]:
dtree = DecisionTreeClassifier()
dtree.fit(X_train, y_train)

## Predictions and Evaluation of Decision Tree

**Create predictions from the test set and create a classification report and a confusion matrix.**

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

dt_predictions = dtree.predict(X_test)

In [ ]:
print('Confusion Matrix - Decision Tree:')
print(confusion_matrix(y_test, dt_predictions))

In [ ]:
print('Classification Report - Decision Tree:')
print(classification_report(y_test, dt_predictions))

## Training the Random Forest Model

Now it's time to train our Random Forest model!

**Create an instance of `RandomForestClassifier` and fit it to the training data.**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier(n_estimators=100, random_state=42)
rfc.fit(X_train, y_train)

## Predictions and Evaluation of Random Forest

**Predict the class of `y` for the X_test data.**

In [ ]:
rfc_predictions = rfc.predict(X_test)

**Create a classification report. Do you notice any difference compared to the Decision Tree?**

In [ ]:
print('Classification Report - Random Forest:')
print(classification_report(y_test, rfc_predictions))

**Show the Confusion Matrix for the Random Forest predictions.**

In [ ]:
print('Confusion Matrix - Random Forest:')
print(confusion_matrix(y_test, rfc_predictions))

## Bonus: Feature Importance

**Let's visualize which features the Random Forest found most important.**

In [ ]:
feature_importances = pd.Series(rfc.feature_importances_, index=X.columns)
top_features = feature_importances.nlargest(15)

plt.figure(figsize=(10,6))
top_features.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 15 Most Important Features (Random Forest)')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()

## Conclusion: Which Model Performed Better?

**The Random Forest outperformed the Decision Tree** in this case.

- The **Decision Tree** achieved an overall accuracy of ~87%, but showed signs of overfitting — it memorizes the training data perfectly but generalizes less well.
- The **Random Forest** achieved ~91% accuracy by combining the predictions of 100 decision trees, reducing variance and improving generalization.

The Random Forest shows higher **precision** for the positive class (clients who subscribed), making it the more reliable model for this classification task.

> In practice, the Random Forest is almost always preferred over a single Decision Tree because it is more robust, less prone to overfitting, and handles noisy data better.

# Great Job!